# Fire Season Timing | Global

In [1]:
'''
Computes fire season timing metrics (onset, peak, end, season length) for all WWF RESOLVE
ecoregions globally for years 2003-2025. Exports daily fire counts to Google Drive as
one CSV per ecoregion per year. Post-run assembly and metric computation happen after
manual download of Drive files.

Data sources:
- MODIS Terra active fire: MODIS/061/MOD14A1
- MODIS Aqua active fire:  MODIS/061/MYD14A1
- Ecoregions:              RESOLVE/ECOREGIONS/2017

Region definition:
- Global — all WWF RESOLVE ecoregions (~847 after removing Rock and Ice)
- Subsetting: use TEST_N / TEST_IDS to run on a reduced set for testing

Output (all paths derived from RUN_LABEL, RUN_VERSION, and BASE_OUT_DIR):
- <BASE_OUT_DIR>/runs/<RUN_LABEL>_<RUN_VERSION>/fire_metrics/daily_counts/  ← downloaded CSVs go here
- <BASE_OUT_DIR>/runs/<RUN_LABEL>_<RUN_VERSION>/fire_metrics/_all_daily_counts.csv
- <BASE_OUT_DIR>/runs/<RUN_LABEL>_<RUN_VERSION>/fire_metrics/_all_metrics.csv
- <BASE_OUT_DIR>/runs/<RUN_LABEL>_<RUN_VERSION>/fire_metrics/_eco_quality.csv
- <BASE_OUT_DIR>/runs/<RUN_LABEL>_<RUN_VERSION>/fire_metrics/master_<RUN_LABEL>_<RUN_VERSION>.csv
- <BASE_OUT_DIR>/runs/<RUN_LABEL>_<RUN_VERSION>/README.txt
- <BASE_OUT_DIR>/runs/<RUN_LABEL>_<RUN_VERSION>/eco_geometries.json
'''

import ee
import pandas as pd
import numpy as np
from diptest import diptest
import os
import time
import datetime as dt
from datetime import datetime, timedelta
import json
import glob
from tqdm import tqdm

In [2]:
# Authenticate and initialize ----------------------------------------------------------------------
ee.Authenticate()
ee.Initialize(project='fire-seasons')

## Run Configuration

In [3]:
# RUN CONFIGURATION --------------------------------------------------------------------------------
# Set these before running anything else. All output paths are derived from these values.

RUN_LABEL   = 'global'  # short name for this run
RUN_VERSION = 'v6'      # increment this for each new run
RUN_NOTES   = """
Global pipeline with single tasks
"""

## Folder setup

In [4]:
# Folder structure and paths -----------------------------------------------------------------------

BASE_OUT_DIR = r'C:\Users\ibekar\Documents\GitProjects\TGPF'  # Windows
# BASE_OUT_DIR = '/Users/ibekar/Github/TGPF'                  # Mac

_run_stamp = dt.date.today().strftime('%Y-%m-%d')
_run_name  = f'{RUN_LABEL}_{RUN_VERSION}'
run_dir    = os.path.join(BASE_OUT_DIR, 'runs', _run_name)
raw_dir    = os.path.join(run_dir, 'raw')
output_dir = os.path.join(run_dir, 'fire_metrics')
daily_dir  = os.path.join(output_dir, 'daily_counts')

os.makedirs(output_dir, exist_ok=True)
os.makedirs(daily_dir, exist_ok=True)

geo_path   = os.path.join(run_dir, 'eco_geometries.json')  # ← add this

print(f'Run name  : {_run_name}')
print(f'Run dir   : {run_dir}')
print(f'Raw dir   : {raw_dir}')
print(f'Output dir: {output_dir}')
print(f'Geo file path: {geo_path}')

Run name  : global_v6
Run dir   : C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v6
Raw dir   : C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v6\raw
Output dir: C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v6\fire_metrics
Geo file path: C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v6\eco_geometries.json


## Data
### MODIS

In [5]:
# LOAD MODIS COLLECTIONS ---------------------------------------------------------------------------
# Terra and Aqua are loaded once here at module level.

terra = (ee.ImageCollection("MODIS/061/MOD14A1")
         .filterDate('2003-01-01', '2025-12-31')
         .select('FireMask'))
aqua  = (ee.ImageCollection("MODIS/061/MYD14A1")
         .filterDate('2003-01-01', '2025-12-31')
         .select('FireMask'))

print('Terra image count:', terra.size().getInfo())
print('Aqua image count:', aqua.size().getInfo())
print('Terra and Aqua collections loaded.')

Terra image count: 8370
Aqua image count: 8384
Terra and Aqua collections loaded.


### Ecoregions

In [6]:
# Load ecoregions ---
# Priority order:
#   1. eco_list already in memory          - skip everything
#   2. eco_geometries.json exists on disk  - load from disk (fast, no GEE)
#   3. Neither                             - fetch from GEE (slow)

ecoregions = ee.FeatureCollection("RESOLVE/ECOREGIONS/2017")

if 'eco_list' in dir() and len(eco_list) > 0:
    print(f'eco_list already in memory ({len(eco_list)} features) — skipping.')

elif os.path.exists(geo_path):
    print(f'Loading eco_list from disk: {geo_path}')
    with open(geo_path, 'r') as f:
        geo_data = json.load(f)

    # Reconstruct eco_list format from saved geometries
    eco_list = []
    for rec in geo_data:
        eco_list.append({
            'properties': {
                'ECO_ID'    : rec['eco_id'],
                'ECO_NAME'  : rec['eco_name'],
                'BIOME_NUM' : rec['biome_num'],
                'BIOME_NAME': rec['biome_name'],
            },
            'geometry': rec['geometry']
        })
    print(f'Loaded {len(eco_list)} ecoregions from disk.')

else:
    n_eco      = ecoregions.size().getInfo()
    batch_size = 100
    offset     = 0
    eco_list   = []

    print(f'Total ecoregions: {n_eco}. Fetching in batches of {batch_size}...')

    while offset < n_eco:
        batch = (ecoregions
                 .select(['ECO_ID', 'ECO_NAME', 'BIOME_NUM', 'BIOME_NAME'])
                 .toList(batch_size, offset)
                 .getInfo())
        eco_list.extend(batch)
        offset += batch_size
        print(f'  Fetched {len(eco_list)} / {n_eco}')

    print(f'Done. {len(eco_list)} ecoregion features loaded.')

# Build eco_records ---
eco_records = []
removed     = []

for f in eco_list:
    p = f['properties']
    if p['ECO_ID'] == 0:
        removed.append({
            'eco_id' : p['ECO_ID']
        })
    else:
        eco_records.append({
            'eco_id'    : p['ECO_ID'],
            'eco_name'  : p['ECO_NAME'],
            'biome_num' : p['BIOME_NUM'],
            'biome_name': p['BIOME_NAME']
            })

print(f'\nBuilt {len(eco_records)} ecoregion records.')
print(f'\n{len(removed)} ecoregion records removed because ECO_ID = 0.')

Loading eco_list from disk: C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v6\eco_geometries.json
Loaded 846 ecoregions from disk.

Built 846 ecoregion records.

0 ecoregion records removed because ECO_ID = 0.


In [7]:
# SAVE GEOMETRIES TO DISK --------------------------------------------------------------------------
# Saves ecoregion geometries as GeoJSON for reuse in visualization notebooks
# Skipped if file already exists.
# Uses geometries already in eco_list, no GEE calls needed

if os.path.exists(geo_path):
    print(f'Geometries already saved. Skipping. ({geo_path})')
else:
    geo_records_export = []
    for f in eco_list:
        p = f['properties']
        if p['ECO_ID'] == 0:
            continue
        geo_records_export.append({
            'eco_id'    : p['ECO_ID'],
            'eco_name'  : p['ECO_NAME'],
            'biome_num' : p['BIOME_NUM'],
            'biome_name': p['BIOME_NAME'],
            'geometry'  : f['geometry']
        })

    with open(geo_path, 'w') as f:
        json.dump(geo_records_export, f)

    print(f'Saved {len(geo_records_export)} geometries → {geo_path}')

Geometries already saved. Skipping. (C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v6\eco_geometries.json)


### Batch complexity

In [8]:
# GEOMETRY COMPLEXITY ANALYSIS ---------------------------------------------------------------------
# Computes vertex count per ecoregion from eco_geometries.json.
# Used to sort eco_run by size so lightweight ecoregions complete first.
#
# Tiers (informational only):
#   low     : < 2,000  vertices
#   medium  : < 50,000 vertices
#   high    : < 250,000 vertices
#   extreme : ≥ 250,000 vertices

COMPLEXITY_THRESHOLDS = {
    'low'    : 2000,
    'medium' : 50000,
    'high'   : 250000,
}

def count_vertices(geometry):
    geo_type = geometry.get('type', '')
    coords   = geometry.get('coordinates', [])
    if geo_type == 'Polygon':
        return sum(len(ring) for ring in coords)
    elif geo_type == 'MultiPolygon':
        return sum(len(ring) for poly in coords for ring in poly)
    return 0

with open(geo_path, 'r') as f:
    geo_data = json.load(f)

complexity_records = []
for rec in geo_data:
    n_vertices = count_vertices(rec['geometry'])
    if n_vertices < COMPLEXITY_THRESHOLDS['low']:
        tier = 'low'
    elif n_vertices < COMPLEXITY_THRESHOLDS['medium']:
        tier = 'medium'
    elif n_vertices < COMPLEXITY_THRESHOLDS['high']:
        tier = 'high'
    else:
        tier = 'extreme'

    complexity_records.append({
        'eco_id'    : rec['eco_id'],
        'eco_name'  : rec['eco_name'],
        'n_vertices': n_vertices,
        'tier'      : tier,
    })

complexity_df = pd.DataFrame(complexity_records)

# Summary
print('Vertex count stats:')
print(complexity_df['n_vertices'].describe().round(0).to_string())
print('Complexity tier summary:')
print(complexity_df['tier'].value_counts().to_string())
print()
print('Most complex ecoregions (top 10):')
print(complexity_df.nlargest(10, 'n_vertices')[['eco_id', 'eco_name', 'n_vertices', 'tier']].to_string(index=False))
print()


Vertex count stats:
count       846.0
mean      13788.0
std       39282.0
min           0.0
25%         800.0
50%        2377.0
75%        7347.0
max      496319.0
Complexity tier summary:
tier
medium     404
low        386
high        51
extreme      5

Most complex ecoregions (top 10):
 eco_id                                   eco_name  n_vertices    tier
     79 Ethiopian montane grasslands and woodlands      496319 extreme
    393           Mid-Atlantic US coastal savannas      342387 extreme
     43                      East Sudanian savanna      317219 extreme
     42                       Dry miombo woodlands      290324 extreme
     39     Central Zambezian wet miombo woodlands      271516 extreme
    125                 North Victoria Land tundra      233854    high
    185                  Einasleigh upland savanna      216624    high
     90                     Renosterveld shrubland      206357    high
    347              Atlantic coastal pine barrens      203684    high
 

In [9]:
# SUBSETTING ---------------------------------------------------------------------------------------
# Set to None to disable 
TEST_N   = None
TEST_IDS = None # [789, 790, 788, 802]

# APPLY SUBSETTING ---------------------------------------------------------------------------------
eco_run = eco_records

if TEST_IDS is not None:
    eco_run = [e for e in eco_run if e['eco_id'] in TEST_IDS]
    print(f'Subsetting to {len(eco_run)} ecoregions by ID: {TEST_IDS}')

if TEST_N is not None:
    eco_run = eco_run[:TEST_N]
    print(f'Subsetting to first {TEST_N} ecoregions.')

print(f'Running pipeline on {len(eco_run)} / {len(eco_records)} ecoregions.')

Running pipeline on 846 / 846 ecoregions.


## Parameters

In [10]:
# General ------------------------------------------------------------------------------------------
FIRE_MASK_MIN   = 8     # FireMask threshold: >= 8 = nominal + high confidence only
ONSET_THRESHOLD = 0.05  # Cumulative fraction threshold for fire season onset (5%)
END_THRESHOLD   = 0.95  # Cumulative fraction threshold for fire season end (95%)
MIN_DETECTIONS  = 20    # Minimum annual fire detections required to compute metrics
YEARS           = list(range(2003, 2026))  # Full study period: 2003–2025

# BIMODALITY DIAGNOSTICS ---------------------------------------------------------------------------
# Applied only when season_length > MIN_SEASON_FOR_BIMODALITY days.
MIN_SEASON_FOR_BIMODALITY = 90    # Minimum season length (days) before bimodality is assessed
BC_THRESHOLD              = 0.555 # Bimodality coefficient above this → bimodality signal
DIP_PVAL_THRESHOLD = 0.05  # dip test p < this → reject unimodality

## README

In [11]:
# WRITE README -------------------------------------------------------------------------------------
_readme_path = os.path.join(run_dir, 'README.txt')
with open(_readme_path, 'w') as _f:
    _f.write(f'Run name    : {_run_name}\n')
    _f.write(f'Date        : {_run_stamp}\n')
    _f.write(f'Years       : {YEARS[0]}–{YEARS[-1]}\n')
    _f.write(f'TEST_N      : {TEST_N}\n')
    _f.write(f'TEST_IDS    : {TEST_IDS}\n')
    _f.write(f'Ecoregions  : {len(eco_run)} / {len(eco_records)}\n')
    _f.write(f'\nNotes:\n{RUN_NOTES.strip()}\n')
print(f'README written → {_readme_path}')

README written → C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v6\README.txt


## Main pipeline

### Helper Function

In [12]:
# HELPER FUNCTIONS ---------------------------------------------------------------------------------
# For each ecoregion × year: stacks all daily images into one multi-band image
# using toBands(), then calls reduceRegion() ONCE per year on the full stack.
# This mirrors the approach proven to work in the Med Basin pipeline.
# Result: 23 reduceRegion() calls per ecoregion (one per year), not 8,400.
#
# Band names are constructed in Python (day_001 ... day_365/366) and known upfront,
# so server-side dict.get() can extract values without iterating in Python.

# Load ecoregions collection once — geometry fetched server-side by ECO_ID
_ecoregions = ee.FeatureCollection("RESOLVE/ECOREGIONS/2017")


def build_fire_fc(eco):
    """
    Builds a server-side GEE FeatureCollection of daily fire detection counts
    for one ecoregion across ALL years, suitable for ee.batch.Export.

    For each year:
      - Builds 365/366 single-band daily fire images
      - Stacks them into one multi-band image via toBands()
      - Calls reduceRegion() ONCE on the stack
      - Extracts per-day counts by known band name keys server-side
      - Returns one ee.Feature per day

    Total reduceRegion() calls: 23 (one per year). Not 8,400.
    """
    eco_id   = eco['eco_id']
    eco_name = eco['eco_name']

    geometry   = _ecoregions.filter(ee.Filter.eq('ECO_ID', eco_id)).geometry()
    empty      = ee.Image.constant(0).rename('FireMask').toUint8()
    years_list = ee.List(YEARS)

    def process_year(year):
        year   = ee.Number(year).toInt()
        start  = ee.Date.fromYMD(year, 1, 1)
        end    = ee.Date.fromYMD(year.add(1), 1, 1)
        n_days = end.difference(start, 'day').toInt()

        terra_year = terra.filterDate(start, end)
        aqua_year  = aqua.filterDate(start, end)
        day_seq    = ee.List.sequence(0, n_days.subtract(1))

        def make_daily_image(d):
            d        = ee.Number(d)
            date     = start.advance(d, 'day')
            date_end = date.advance(1, 'day')

            terra_day = terra_year.filterDate(date, date_end)
            aqua_day  = aqua_year.filterDate(date, date_end)

            t = ee.Image(ee.Algorithms.If(
                terra_day.size().gt(0),
                terra_day.select('FireMask').max(),
                empty
            ))
            a = ee.Image(ee.Algorithms.If(
                aqua_day.size().gt(0),
                aqua_day.select('FireMask').max(),
                empty
            ))

            fire_binary = t.max(a).gte(FIRE_MASK_MIN).unmask(0)

            band_name = ee.String('day_').cat(
                d.add(1).toInt().format('%03d')
            )

            return fire_binary.rename(band_name)

        # Stack all daily images into one multi-band image
        stacked = ee.ImageCollection(day_seq.map(make_daily_image)).toBands()

        # ONE reduceRegion call on the full stack
        counts_dict = stacked.reduceRegion(
            reducer   = ee.Reducer.sum(),
            geometry  = geometry,
            scale     = 1000,
            maxPixels = 1e9,
            bestEffort= True
        )

        # Extract per-day counts by known band keys server-side.
        # toBands() prepends the image index to the band name: '0_day_001', '1_day_002' etc.
        # We reconstruct those keys from the known day sequence.
        def extract_day_feature(d):
            d         = ee.Number(d)
            doy       = d.add(1).toInt()
            band_key  = d.toInt().format('%d').cat('_day_').cat(doy.format('%03d'))
            count     = ee.Number(counts_dict.get(band_key)).toInt()

            return ee.Feature(None, {
                'eco_id'      : eco_id,
                'eco_name'    : eco_name,
                'year'        : year,
                'doy'         : doy,
                'n_detections': count
            })

        return ee.FeatureCollection(day_seq.map(extract_day_feature))

    return ee.FeatureCollection(years_list.map(process_year)).flatten()

### GEE submission

In [13]:
# SORT eco_run BY GEOMETRY SIZE (smallest first) ---------------------------------------------------
# Ensures lightweight ecoregions complete early, giving usable data while
# heavy ecoregions (extreme tier) are still running at the end of the queue.

eco_run = sorted(
    eco_run,
    key=lambda e: complexity_df.loc[complexity_df['eco_id'] == e['eco_id'], 'n_vertices'].values[0]
)

# Confirm order
print('eco_run sorted by n_vertices ascending.')
print()
print('First 5 (smallest):')
for e in eco_run[:5]:
    v = complexity_df.loc[complexity_df['eco_id'] == e['eco_id'], 'n_vertices'].values[0]
    print(f'  eco_id={e["eco_id"]:4d} | {v:>8,} vertices | {e["eco_name"]}')

print()
print('Last 5 (largest):')
for e in eco_run[-5:]:
    v = complexity_df.loc[complexity_df['eco_id'] == e['eco_id'], 'n_vertices'].values[0]
    print(f'  eco_id={e["eco_id"]:4d} | {v:>8,} vertices | {e["eco_name"]}')

eco_run sorted by n_vertices ascending.

First 5 (smallest):
  eco_id=  97 |        0 vertices | Kalahari xeric savanna
  eco_id=  76 |        0 vertices | Zambezian flooded grasslands
  eco_id= 198 |        0 vertices | Esperance mallee
  eco_id= 202 |        0 vertices | Jarrah-Karri forest and shrublands
  eco_id=  88 |        0 vertices | Albany thickets

Last 5 (largest):
  eco_id=  39 |  271,516 vertices | Central Zambezian wet miombo woodlands
  eco_id=  42 |  290,324 vertices | Dry miombo woodlands
  eco_id=  43 |  317,219 vertices | East Sudanian savanna
  eco_id= 393 |  342,387 vertices | Mid-Atlantic US coastal savannas
  eco_id=  79 |  496,319 vertices | Ethiopian montane grasslands and woodlands


In [14]:
RUN_THIS = False
if not RUN_THIS:
    raise RuntimeError("Set RUN_THIS=True if you really want to execute this")

# SUBMIT FIRE EXPORT TASKS -------------------------------------------------------------------------
# One task per ecoregion, all years, using toBands() + single reduceRegion() per year.
# 846 tasks submitted sequentially with a short sleep between each to avoid API rate limits.
#
# Naming convention:
#   FIRE_<RUN_ID>_<run_name>_eco_<eco_id>
#
# When complete: move all CSVs from Drive/<DRIVE_FOLDER>/ to:
#   <output_dir>/daily_counts/
#
# Monitor at: https://code.earthengine.google.com/tasks

DRIVE_FOLDER = f'fire_{_run_name}_daily'
RUN_ID       = dt.date.today().strftime('%Y%m%d')

os.makedirs(output_dir, exist_ok=True)
os.makedirs(daily_dir,  exist_ok=True)

submitted_log = os.path.join(output_dir, '_submitted_tasks.txt')
failed_log    = os.path.join(output_dir, '_failed_tasks.txt')

# Reset logs
with open(submitted_log, 'w') as log:
    log.write(f'Submission started: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}\n')
    log.write(f'Run: {_run_name} | RUN_ID: {RUN_ID}\n')
    log.write(f'{"-" * 60}\n')

with open(failed_log, 'w') as log:
    log.write(f'Failed tasks log initialized: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}\n')
    log.write(f'Run: {_run_name} | RUN_ID: {RUN_ID}\n')
    log.write(f'{"-" * 60}\n')

n_submitted = 0
n_failed    = 0

for eco in tqdm(eco_run, desc='Submitting'):
    eco_id    = eco['eco_id']
    eco_name  = eco['eco_name']
    safe_name = eco_name.replace(' ', '_').replace('/', '_')

    task_desc = f'FIRE_{RUN_ID}_{_run_name}_eco_{eco_id}'
    file_name = f'eco_{eco_id}_{safe_name}_daily'

    try:
        daily_fc = build_fire_fc(eco)

        task = ee.batch.Export.table.toDrive(
            collection    = daily_fc,
            description   = task_desc,
            folder        = DRIVE_FOLDER,
            fileNamePrefix= file_name,
            fileFormat    = 'CSV',
            selectors     = ['eco_id', 'eco_name', 'year', 'doy', 'n_detections']
        )
        task.start()
        n_submitted += 1

        with open(submitted_log, 'a') as log:
            log.write(f'{eco_id} | {eco_name} | {task_desc}\n')

    except Exception as e:
        n_failed += 1
        print(f'  WARNING: {eco_name} ({eco_id}) failed to submit — {e}')
        with open(failed_log, 'a') as log:
            log.write(f'{eco_id} | {eco_name} | {str(e)}\n')

    time.sleep(1)

print(f'\nSubmission complete.')
print(f'  Submitted : {n_submitted}')
print(f'  Failed    : {n_failed}')
print(f'\nMonitor at: https://code.earthengine.google.com/tasks')
print(f'\nWhen complete, move CSVs from Drive/{DRIVE_FOLDER}/ to:')
print(f'  {daily_dir}')

RuntimeError: Set RUN_THIS=True if you really want to execute this

## Monitoring

In [16]:

# NOTE: RUN_ID must match the value used at submission time.
# If the kernel was restarted or it is a different day, manually set:
#   RUN_ID = 'YYYYMMDD'  ← use the date string from _submitted_tasks.txt

# SET THESE IF KERNEL WAS RESTARTED -------------------------------------------
RUN_ID   = '20260416'   # ← date from _submitted_tasks.txt
_run_name = 'global_v6' # ← must match submission run
# -----------------------------------------------------------------------------

now = datetime.now()

tasks    = ee.data.getTaskList()
my_tasks = [t for t in tasks
            if t['description'].startswith(f'FIRE_{RUN_ID}_{_run_name}')
            and '_test' not in t['description']]

ready     = sum(1 for t in my_tasks if t['state'] == 'READY')
running   = sum(1 for t in my_tasks if t['state'] == 'RUNNING')
completed = sum(1 for t in my_tasks if t['state'] == 'COMPLETED')
failed    = sum(1 for t in my_tasks if t['state'] == 'FAILED')

# Average compute time from GEE task metadata
completed_tasks = [t for t in my_tasks if t['state'] == 'COMPLETED']

if completed_tasks:
    runtimes_min = []
    for t in completed_tasks:
        start_ms  = t.get('start_timestamp_ms', None)
        update_ms = t.get('update_timestamp_ms', None)
        if start_ms and update_ms:
            runtimes_min.append((update_ms - start_ms) / 60000)

    if runtimes_min:
        avg_min_per_task   = sum(runtimes_min) / len(runtimes_min)
        remaining_tasks    = ready + running
        eta_minutes        = avg_min_per_task * remaining_tasks
        eta_hours, eta_rem = divmod(eta_minutes, 60)
        eta_datetime       = now + timedelta(minutes=eta_minutes)
        eta_str = (
            f'avg {avg_min_per_task:.1f} min/task '
            f'| ETA ~{int(eta_hours)}h {int(eta_rem)}m '
            f'| est. done: {eta_datetime.strftime("%Y-%m-%d %H:%M")}'
        )
    else:
        eta_str = 'avg — | ETA — | est. done: —'
else:
    eta_str = 'avg — | ETA — | est. done: —'

print(f'Checked   : {now.strftime("%Y-%m-%d %H:%M:%S")}')
print(f'READY     : {ready}')
print(f'RUNNING   : {running}')
print(f'COMPLETED : {completed}')
print(f'FAILED    : {failed}')
print(f'{eta_str}')

Checked   : 2026-04-23 14:27:16
READY     : 0
RUNNING   : 0
COMPLETED : 793
FAILED    : 53
avg 4.5 min/task | ETA ~0h 0m | est. done: 2026-04-23 14:27


### Failed Tasks

In [ ]:
# IDENTIFY FAILED GEE TASKS -----------------------------------------------------------------------
# Queries the GEE task list for server-side failures from this run.
# Uses the task description naming convention to extract eco_ids.
#
# Requires RUN_ID and _run_name to be set (from monitoring cell).

task_log = ee.data.getTaskList()
filtered_log = [t for t in task_log
                if t['description'].startswith(f'FIRE_{RUN_ID}_{_run_name}')
                and '_test' not in t['description']]

failed_log = [t for t in filtered_log if t['state'] == 'FAILED']

failed_eco_ids = list(set(
    int(t['description'].split('_eco_')[-1]) for t in failed_log
))
failed_eco_ids.sort()

if not failed_eco_ids:
    print('\nNo failed tasks — nothing to resubmit.')
else:
    # Show which ecoregions failed, with names from eco_records
    eco_lookup = {e['eco_id']: e['eco_name'] for e in eco_records}
    print(f'\nFailed ecoregions:')
    for eid in failed_eco_ids:
        name = eco_lookup.get(eid, '???')
        print(f'  eco_id={eid:4d} | {name}')

print()
print(f'Run: {_run_name} | RUN_ID: {RUN_ID}')
print(f'Total tasks found : {len(filtered_log)}')
print(f'Failed tasks      : {len(failed_log)}')
print(f'Unique eco_ids    : {len(failed_eco_ids)}')


Failed ecoregions:
  eco_id=  56 | South Arabian fog woodlands, shrublands, and dune
  eco_id=  57 | Southern Acacia-Commiphora bushlands and thickets
  eco_id=  61 | Victoria Basin forest-savanna
  eco_id=  81 | Highveld grasslands
  eco_id=  94 | Gariep Karoo
  eco_id= 134 | Transantarctic Mountains tundra
  eco_id= 150 | Queensland tropical rain forests
  eco_id= 152 | Solomon Islands rain forests
  eco_id= 161 | Vogelkop-Aru lowland rain forests
  eco_id= 191 | Eastern Australia mulga shrublands
  eco_id= 216 | Western Australian Mulga shrublands
  eco_id= 220 | Borneo montane rain forests
  eco_id= 243 | Maldives-Lakshadweep-Chagos Archipelago tropical moist forests
  eco_id= 256 | Northern Indochina subtropical forests
  eco_id= 279 | Sumatran montane rain forests
  eco_id= 295 | Khathiar-Gir dry deciduous forests
  eco_id= 339 | Northeast US Coastal forests
  eco_id= 345 | Alberta-British Columbia foothills forests
  eco_id= 382 | Southern Hudson Bay taiga
  eco_id= 384 | Weste

In [ ]:
# RESUBMIT FAILED TASKS ---------------------------------------------------------------------------
# Resubmits only the ecoregions that failed on the GEE server side.
# Uses the same build_fire_fc() and export logic as the original submission cell.
#
# Requires: failed_eco_ids (from the identification cell above)
#           eco_records, build_fire_fc, _run_name, output_dir, daily_dir

RUN_THIS_RETRY = False
if not RUN_THIS_RETRY:
    raise RuntimeError("Set RUN_THIS_RETRY=True if you really want to resubmit failed tasks")

if not failed_eco_ids:
    raise RuntimeError("No failed eco_ids found — nothing to resubmit.")

# Step 1 — Filter to failed ecoregions
failed_eco_run = [e for e in eco_records if e['eco_id'] in failed_eco_ids]
print(f'Resubmitting {len(failed_eco_run)} / {len(eco_records)} ecoregions.')

# Step 2 — Retry identifiers
RETRY_ID     = dt.date.today().strftime('%Y%m%d')
DRIVE_FOLDER = f'fire_{_run_name}_daily'  # same folder — filenames are unique per eco_id

# Step 3 — Submit
n_submitted = 0
n_failed    = 0

for eco in tqdm(failed_eco_run, desc='Resubmitting'):
    eco_id    = eco['eco_id']
    eco_name  = eco['eco_name']
    safe_name = eco_name.replace(' ', '_').replace('/', '_')

    task_desc = f'FIRE_{RETRY_ID}_{_run_name}_retry_eco_{eco_id}'
    file_name = f'eco_{eco_id}_{safe_name}_daily'

    try:
        daily_fc = build_fire_fc(eco)

        task = ee.batch.Export.table.toDrive(
            collection     = daily_fc,
            description    = task_desc,
            folder         = DRIVE_FOLDER,
            fileNamePrefix = file_name,
            fileFormat     = 'CSV',
            selectors      = ['eco_id', 'eco_name', 'year', 'doy', 'n_detections']
        )
        task.start()
        n_submitted += 1

    except Exception as e:
        n_failed += 1
        print(f'  WARNING: {eco_name} ({eco_id}) failed to submit — {e}')

    time.sleep(1)

print(f'\nRetry submission complete.')
print(f'  Submitted : {n_submitted}')
print(f'  Failed    : {n_failed}')
print(f'\nMonitor at: https://code.earthengine.google.com/tasks')

In [19]:
# IDENTIFY FAILED GEE TASKS -----------------------------------------------------------------------
# Queries the GEE task list for server-side failures from this run.
# Uses the task description naming convention to extract eco_ids.
#
# Requires RUN_ID and _run_name to be set (from monitoring cell).

RETRY_ID = '20260420'  # or whatever date you ran the retry
_run_name = 'global_v6'  # or whatever your run name is

task_log = ee.data.getTaskList()
filtered_log = [t for t in task_log
                if t['description'].startswith(f'FIRE_{RETRY_ID}_{_run_name}')
                and '_test' not in t['description']]

failed_log = [t for t in filtered_log if t['state'] == 'FAILED']

failed_eco_ids = list(set(
    int(t['description'].split('_eco_')[-1]) for t in failed_log
))
failed_eco_ids.sort()

if not failed_eco_ids:
    print('\nNo failed tasks — nothing to resubmit.')
else:
    # Show which ecoregions failed, with names from eco_records
    eco_lookup = {e['eco_id']: e['eco_name'] for e in eco_records}
    print(f'\nFailed ecoregions:')
    for eid in failed_eco_ids:
        name = eco_lookup.get(eid, '???')
        print(f'  eco_id={eid:4d} | {name}')

print()
print(f'Run: {_run_name} | RUN_ID: {RUN_ID}')
print(f'Total tasks found : {len(filtered_log)}')
print(f'Failed tasks      : {len(failed_log)}')
print(f'Unique eco_ids    : {len(failed_eco_ids)}')


Failed ecoregions:
  eco_id=  57 | Southern Acacia-Commiphora bushlands and thickets
  eco_id=  81 | Highveld grasslands
  eco_id=  94 | Gariep Karoo
  eco_id= 134 | Transantarctic Mountains tundra
  eco_id= 150 | Queensland tropical rain forests
  eco_id= 216 | Western Australian Mulga shrublands
  eco_id= 220 | Borneo montane rain forests
  eco_id= 295 | Khathiar-Gir dry deciduous forests
  eco_id= 339 | Northeast US Coastal forests
  eco_id= 345 | Alberta-British Columbia foothills forests
  eco_id= 394 | Montana Valley and Foothill grasslands
  eco_id= 417 | Kalaallit Nunaat Arctic steppe
  eco_id= 429 | Colorado Plateau shrublands
  eco_id= 430 | Great Basin shrub steppe
  eco_id= 435 | Sonoran desert
  eco_id= 498 | Rio Negro campinarana
  eco_id= 500 | Serra do Mar coastal forests
  eco_id= 507 | Tapajós-Xingu moist forests
  eco_id= 545 | Sinaloan dry forests
  eco_id= 574 | Uruguayan savanna
  eco_id= 642 | Guizhou Plateau broadleaf and mixed forests
  eco_id= 772 | Chukchi P

## Post-Run Assembly

### Function: compute timing metrics

In [13]:
def compute_timing_metrics(df, year):
    total = df['n_detections'].sum()
    if total < MIN_DETECTIONS:
        return None

    df     = df.copy().sort_values('doy').reset_index(drop=True)
    doys   = df['doy'].values.astype(float)
    counts = df['n_detections'].values.astype(float)

    cumulative = df['n_detections'].cumsum()
    cum_frac   = cumulative / total

    def doy_at_frac(frac):
        rows = df[cum_frac >= frac]
        return int(rows.iloc[0]['doy']) if not rows.empty else None

    # 1. Primary metrics
    onset_doy     = doy_at_frac(ONSET_THRESHOLD)
    end_doy       = doy_at_frac(END_THRESHOLD)
    rolling       = df['n_detections'].rolling(7, center=True, min_periods=1).mean()
    peak_doy      = int(df.loc[rolling.idxmax(), 'doy'])
    season_length = (end_doy - onset_doy + 1) if (onset_doy and end_doy) else None

    def doy_to_month(doy, yr):
        if doy is None:
            return None
        return (pd.Timestamp(year=yr, month=1, day=1) + pd.Timedelta(days=doy - 1)).month

    onset_month = doy_to_month(onset_doy, year)
    peak_month  = doy_to_month(peak_doy,  year)

    peak_outside_window = (
        (onset_doy is not None and end_doy is not None) and
        not (onset_doy <= peak_doy <= end_doy)
    )

    # 2. Alternative thresholds
    onset_doy_10 = doy_at_frac(0.10)
    end_doy_90   = doy_at_frac(0.90)
    onset_doy_15 = doy_at_frac(0.15)
    end_doy_85   = doy_at_frac(0.85)

    # 3. Profile shape
    season_mask        = (df['doy'] >= onset_doy) & (df['doy'] <= end_doy)
    active_days        = int((df.loc[season_mask, 'n_detections'] > 0).sum())
    conc_mask          = (df['doy'] >= peak_doy - 45) & (df['doy'] <= peak_doy + 45)
    peak_concentration = round(float(df.loc[conc_mask, 'n_detections'].sum() / total), 4)

    w_mean = (doys * counts).sum() / counts.sum()
    diffs  = doys - w_mean
    w_var  = (counts * diffs**2).sum() / counts.sum()
    w_std  = np.sqrt(w_var)

    median_doy      = doy_at_frac(0.50)
    mean_median_div = (w_mean - median_doy) if median_doy is not None else None
    q25_doy         = doy_at_frac(0.25)
    q75_doy         = doy_at_frac(0.75)
    iqr_season_length = (q75_doy - q25_doy + 1) if (q25_doy and q75_doy) else None

    if w_std > 0:
        w_skewness     = round(float((counts * (diffs / w_std)**3).sum() / counts.sum()), 4)
        w_kurtosis_raw = round(float((counts * (diffs / w_std)**4).sum() / counts.sum()), 4)
    else:
        w_skewness     = None
        w_kurtosis_raw = None

    # 4. Bimodality — two independent screens: BC and Hartigan's dip test
    bc                = None
    bimodal_flag_bc   = 0
    dip_stat          = None
    dip_pval          = None
    bimodal_flag_dip  = 0
    bimodal_flag_year = 0  # consensus: AND of BC and dip flags

    if season_length and season_length > MIN_SEASON_FOR_BIMODALITY:
        n = len(counts)
        if (w_std > 0 and n > 3
                and w_skewness is not None and w_kurtosis_raw is not None):
            excess_kurtosis = w_kurtosis_raw - 3
            correction      = 3 * ((n - 1) ** 2) / ((n - 2) * (n - 3))
            bc = round(float((w_skewness ** 2 + 1) / (excess_kurtosis + correction)), 4)

        if bc is not None and bc > BC_THRESHOLD:
            bimodal_flag_bc = 1

        try:
            expanded = np.repeat(doys, counts.astype(int))
            if len(expanded) > 72000:
                expanded = np.random.choice(expanded, size=72000, replace=False)
            if len(expanded) >= 4:
                dip_stat, dip_pval = diptest(expanded)
                dip_stat = round(float(dip_stat), 4)
                dip_pval = round(float(dip_pval), 4)
                if dip_pval < DIP_PVAL_THRESHOLD:
                    bimodal_flag_dip = 1
        except Exception:
            pass

        bimodal_flag_year = 1 if (bimodal_flag_bc and bimodal_flag_dip) else 0

    # -----------------------------------------------------------------------
    # CIRCULAR METRICS — rotate-then-linear
    #
    # 1) Compute the circular mean of the year's fire activity.
    # 2) Shift the DOY axis so circ_mean_doy lands at DOY 183 (mid-year).
    #    This puts the dead zone at the ends of the rotated axis, exactly
    #    where the linear 5%/95% rule expects empty shoulders.
    # 3) Apply the linear cumulative rule on the rotated axis.
    # 4) Rotate onset/end back to calendar DOY. Length taken directly on
    #    the rotated axis — no wrap logic needed.
    # -----------------------------------------------------------------------

    # Step 1 — circular mean + resultant length
    angles   = 2 * np.pi * (doys - 1) / 365
    sin_sum  = np.sum(counts * np.sin(angles))
    cos_sum  = np.sum(counts * np.cos(angles))
    circ_R   = round(float(np.sqrt(sin_sum**2 + cos_sum**2) / total), 4)

    mean_angle        = np.arctan2(sin_sum, cos_sum)
    circ_mean_doy_raw = (mean_angle * 365 / (2 * np.pi)) + 1
    if circ_mean_doy_raw < 1:
        circ_mean_doy_raw += 365
    circ_mean_doy = int(round(circ_mean_doy_raw))

    # Coherence threshold — below this, the circular mean is unstable
    # and onset/end cannot be reliably estimated from the circular method
    CIRC_R_MIN    = 0.55
    BOUNDARY_ZONE = 60  # DOYs within this many days of Jan 1 / Dec 31

    near_boundary = (circ_mean_doy < BOUNDARY_ZONE) or \
                    (circ_mean_doy > (365 - BOUNDARY_ZONE))
    circ_rotated  = 1 if (near_boundary and circ_R >= CIRC_R_MIN) else 0

    # Step 2 — rotate-then-linear
    onset_doy_circ  = None
    end_doy_circ    = None
    season_len_circ = None

    
    # Shift so that circ_mean_doy lands at DOY 183
    shift = 183 - circ_mean_doy

    # Relabel every DOY. Peak is now in the middle of the rotated
    # axis; dead zone sits at the two ends.
    rotated_doys = ((doys - 1 + shift) % 365) + 1

    sort_idx       = np.argsort(rotated_doys)
    sorted_rotated = rotated_doys[sort_idx]
    sorted_counts  = counts[sort_idx]

    # Linear 5%/95% on the rotated axis
    cum_frac_rot = np.cumsum(sorted_counts) / total
    onset_mask   = cum_frac_rot >= ONSET_THRESHOLD
    end_mask     = cum_frac_rot >= END_THRESHOLD

    if onset_mask.any() and end_mask.any():
        onset_rotated = float(sorted_rotated[onset_mask][0])
        end_rotated   = float(sorted_rotated[end_mask][0])

        # Season length — directly on rotated axis (guaranteed ordered)
        season_len_circ = int(end_rotated - onset_rotated + 1)

        # Rotate onset/end back to calendar DOY
        onset_doy_circ = int(((onset_rotated - 1 - shift) % 365) + 1)
        end_doy_circ   = int(((end_rotated   - 1 - shift) % 365) + 1)

    return {
        'onset_doy'           : onset_doy,
        'peak_doy'            : peak_doy,
        'end_doy'             : end_doy,
        'season_length'       : season_length,
        'n_detections'        : int(total),
        'onset_month'         : onset_month,
        'peak_month'          : peak_month,
        'peak_outside_window' : int(peak_outside_window),
        'onset_doy_10'        : onset_doy_10,
        'end_doy_90'          : end_doy_90,
        'onset_doy_15'        : onset_doy_15,
        'end_doy_85'          : end_doy_85,
        'median_doy'          : median_doy,
        'mean_median_div'     : mean_median_div,
        'q25_doy'             : q25_doy,
        'q75_doy'             : q75_doy,
        'iqr_season_length'   : iqr_season_length,
        'active_days'         : active_days,
        'peak_concentration'  : peak_concentration,
        'skewness'            : w_skewness,
        'kurtosis_raw'        : w_kurtosis_raw,
        'bc'                  : bc,
        'bimodal_flag_bc'     : bimodal_flag_bc,
        'dip_stat'            : dip_stat,
        'dip_pval'            : dip_pval,
        'bimodal_flag_dip'    : bimodal_flag_dip,
        'bimodal_flag_year'   : bimodal_flag_year,
        # Circular metrics
        'circ_mean_doy'   : circ_mean_doy,
        'circ_R'          : circ_R,
        'circ_rotated'    : circ_rotated,
        'onset_doy_circ'  : onset_doy_circ,
        'end_doy_circ'    : end_doy_circ,
        'season_len_circ' : season_len_circ,
    }

### Avengers... Assemble

In [14]:
# POST-RUN ASSEMBLY:  DAILY COUNTS + METRICS -------------------------------------------------------
# Reads all CSVs from daily_counts/, concatenates into one daily file,
# then runs compute_timing_metrics() per ecoregion per year.
#
# Run this after all Drive files have been moved to daily_counts/.
# Ensure daily_counts/ folder is clean before running no duplicate files.

YEARS = list(range(2003, 2026)) # safety for kernel crashes 

# Assemble daily counts ---
daily_files = sorted(glob.glob(os.path.join(daily_dir, '[!_]*_daily.csv')))

if not daily_files:
    print('No daily CSV files found. Have you moved the Drive exports to daily_counts/?')
else:
    print(f'Found {len(daily_files)} CSV files. Loading...')

    daily_combined = pd.concat(
        [pd.read_csv(f) for f in daily_files], ignore_index=True
    )

    # Sanity check: flag duplicate eco_id + year + doy rows
    dupes = daily_combined.duplicated(subset=['eco_id', 'year', 'doy']).sum()
    if dupes > 0:
        print(f'WARNING: {dupes} duplicate eco_id/year/doy rows — check for duplicate files')
    else:
        print('No duplicates found.')

    daily_combined.to_csv(os.path.join(output_dir, '_all_daily_counts.csv'), index=False)
    print(f'Daily counts: {len(daily_files)} files → {len(daily_combined)} rows '
          f'across {daily_combined["eco_id"].nunique()} ecoregions')

    # Compute metrics per ecoregion per year ---
    all_metrics  = []
    failed_years = []

    if 'eco_records' in locals():
        eco_meta_lookup = {e['eco_id']: e for e in eco_records}
        print('eco_meta_lookup built from eco_records in memory.')
    else:
        with open(geo_path, 'r') as f:
            geo_data = json.load(f)
        eco_meta_lookup = {rec['eco_id']: rec for rec in geo_data}
        print(f'eco_records not in memory — loaded eco_meta_lookup from {geo_path}')

    for eco_id, eco_daily in daily_combined.groupby('eco_id'):
        eco_name   = eco_daily['eco_name'].iloc[0]
        biome_num  = None
        biome_name = None

        match = eco_meta_lookup.get(eco_id)
        if match:
            biome_num  = match['biome_num']
            biome_name = match['biome_name']

        eco_metrics = []

        for year, year_group in eco_daily.groupby('year'):
            df_year = year_group[['doy', 'n_detections']].copy()
            metrics = compute_timing_metrics(df_year, year)

            if metrics is not None:
                metrics['eco_id']    = eco_id
                metrics['eco_name']  = eco_name
                metrics['biome_num'] = biome_num
                metrics['biome_name']= biome_name
                metrics['year']      = year
                eco_metrics.append(metrics)
                all_metrics.append(metrics)
            else:
                failed_years.append({
                    'eco_id': eco_id, 'eco_name': eco_name,
                    'year': year, 'reason': 'insufficient_detections'
                })

        n_years_valid   = len(eco_metrics)
        pct_years_valid = round(n_years_valid / len(YEARS), 3)
        for m in eco_metrics:
            m['n_years_valid']   = n_years_valid
            m['pct_years_valid'] = pct_years_valid

        if eco_metrics:
            safe_name = eco_name.replace(' ', '_').replace('/', '_')
            eco_path  = os.path.join(output_dir, f'{eco_id}_{safe_name}.csv')
            pd.DataFrame(eco_metrics).to_csv(eco_path, index=False)

    # Save combined metrics
    if all_metrics:
        metrics_combined = pd.DataFrame(all_metrics)
        metrics_combined.to_csv(os.path.join(output_dir, '_all_metrics.csv'), index=False)
        print(f'Metrics: {len(all_metrics)} ecoregion-year rows across '
              f'{metrics_combined["eco_id"].nunique()} ecoregions')

    # Save failed log
    if failed_years:
        pd.DataFrame(failed_years).to_csv(
            os.path.join(output_dir, '_failed.csv'), index=False
        )
        print(f'Failed years: {len(failed_years)} → _failed.csv')

Found 793 CSV files. Loading...
No duplicates found.
Daily counts: 793 files → 6661993 rows across 793 ecoregions
eco_records not in memory — loaded eco_meta_lookup from C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v6\eco_geometries.json
Metrics: 14885 ecoregion-year rows across 707 ecoregions
Failed years: 3354 → _failed.csv


### Ecoregion-level metrics

In [15]:
# ECOREGION-LEVEL QUALITY METRICS ------------------------------------------------------------------
# Computed across years per ecoregion from the assembled files.
# Items: CV of peak DOY, interannual profile correlation, ecoregion bimodal flags.
#
# bimodal_consensus_eco encodes agreement between BC and dip test at the ecoregion level:
#   2 = strong  — both BC and dip flag the ecoregion (both >30% of years flagged)
#   1 = weak    — only one method flags it
#   0 = clean   — neither method flags it
#
# Reads from: _all_metrics.csv and _all_daily_counts.csv
# Writes to:  _eco_quality.csv

metrics_df = pd.read_csv(os.path.join(output_dir, '_all_metrics.csv'))
daily_df   = pd.read_csv(os.path.join(output_dir, '_all_daily_counts.csv'))

print(f'Loaded {len(metrics_df)} ecoregion-year rows across '
      f'{metrics_df["eco_id"].nunique()} ecoregions.')

# -----------------------------------------------------------------------
# CV of peak DOY across years
# -----------------------------------------------------------------------
cv_df = (
    metrics_df.groupby('eco_id')['peak_doy']
    .agg(cv_peak_doy=lambda x: round(float(x.std() / x.mean()), 4)
                                if len(x) > 1 and x.mean() != 0 else None)
    .reset_index()
)

# -----------------------------------------------------------------------
# Ecoregion-level bimodal flags — BC, dip, OR-consensus, consensus strength
# -----------------------------------------------------------------------
def eco_bimodal_flag(flags):
    frac_flagged = (flags == 1).sum() / len(flags)
    return 1 if frac_flagged > 0.50 else 0

flag_df = (
    metrics_df.groupby('eco_id')['bimodal_flag_year']
    .agg(
        frac_flagged     = lambda x: round(float((x == 1).sum() / len(x)), 3),
        bimodal_flag_eco = eco_bimodal_flag
    )
    .reset_index()
)

# Per-method ecoregion flags — saved as columns now, not just scalars
bc_flag_per_eco = (
    metrics_df.groupby('eco_id')['bimodal_flag_bc']
    .apply(eco_bimodal_flag)
    .reset_index()
    .rename(columns={'bimodal_flag_bc': 'bc_flag_eco'})
)

dip_flag_per_eco = (
    metrics_df.groupby('eco_id')['bimodal_flag_dip']
    .apply(eco_bimodal_flag)
    .reset_index()
    .rename(columns={'bimodal_flag_dip': 'dip_flag_eco'})
)

flag_df = flag_df.merge(bc_flag_per_eco,  on='eco_id')
flag_df = flag_df.merge(dip_flag_per_eco, on='eco_id')

# Consensus strength: sum of the two per-method flags → 0, 1, or 2
flag_df['bimodal_consensus_eco'] = flag_df['bc_flag_eco'] + flag_df['dip_flag_eco']

# Summary print
print(f'\nEcoregion bimodal flag summary:')
print(f'  BC flagged  (>30% of years) : {flag_df["bc_flag_eco"].sum()}')
print(f'  Dip flagged (>30% of years) : {flag_df["dip_flag_eco"].sum()}')
print(f'  Consensus clean   (0)       : {(flag_df["bimodal_consensus_eco"] == 0).sum()}')
print(f'  Consensus weak    (1)       : {(flag_df["bimodal_consensus_eco"] == 1).sum()}')
print(f'  Consensus strong  (2)       : {(flag_df["bimodal_consensus_eco"] == 2).sum()}')

# -----------------------------------------------------------------------
# Interannual profile correlation
# -----------------------------------------------------------------------
profile_corr_rows = []

for eco_id, eco_daily in daily_df.groupby('eco_id'):

    pivot = eco_daily.pivot_table(
        index='year', columns='doy',
        values='n_detections', fill_value=0
    )

    if len(pivot) < 3:
        profile_corr_rows.append({'eco_id': eco_id, 'mean_profile_corr': None})
        continue

    mean_profile = pivot.mean(axis=0).values

    corrs = []
    for yr in pivot.index:
        yr_profile = pivot.loc[yr].values
        if yr_profile.sum() > 0 and mean_profile.sum() > 0:
            r = float(np.corrcoef(yr_profile, mean_profile)[0, 1])
            if not np.isnan(r):
                corrs.append(r)

    mean_corr = round(float(np.mean(corrs)), 4) if corrs else None
    profile_corr_rows.append({'eco_id': eco_id, 'mean_profile_corr': mean_corr})

corr_df = pd.DataFrame(profile_corr_rows)

# -----------------------------------------------------------------------
# MERGE AND SAVE
# -----------------------------------------------------------------------
eco_quality = (
    cv_df
    .merge(flag_df,  on='eco_id')
    .merge(corr_df,  on='eco_id')
)

eco_quality_path = os.path.join(output_dir, '_eco_quality.csv')
eco_quality.to_csv(eco_quality_path, index=False)

print(f'\nEcoregion quality metrics saved: {len(eco_quality)} ecoregions')
print(f'Path: {os.path.abspath(eco_quality_path)}')
print()
print(eco_quality[[
    'eco_id', 'cv_peak_doy', 'bc_flag_eco', 'dip_flag_eco',
    'bimodal_consensus_eco', 'frac_flagged', 'mean_profile_corr'
]].head(10).to_string())

Loaded 14885 ecoregion-year rows across 707 ecoregions.

Ecoregion bimodal flag summary:
  BC flagged  (>30% of years) : 415
  Dip flagged (>30% of years) : 608
  Consensus clean   (0)       : 90
  Consensus weak    (1)       : 211
  Consensus strong  (2)       : 406

Ecoregion quality metrics saved: 707 ecoregions
Path: C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v6\fire_metrics\_eco_quality.csv

   eco_id  cv_peak_doy  bc_flag_eco  dip_flag_eco  bimodal_consensus_eco  frac_flagged  mean_profile_corr
0       1       0.1135            0             1                      1         0.174             0.6949
1       2       0.7817            1             1                      2         1.000             0.6753
2       3       0.0709            0             1                      1         0.130             0.5117
3       4       0.1974            0             1                      1         0.333             0.2239
4       5       0.5352            1             1         

## Master CSV file

In [16]:
# Reads from _all_metrics.csv and _eco_quality.csv (both on disk).
# Ecoregion-level quality columns are broadcast to every row for that ecoregion.

master_df   = pd.read_csv(os.path.join(output_dir, '_all_metrics.csv'))
eco_quality = pd.read_csv(os.path.join(output_dir, '_eco_quality.csv'))

master_df = master_df.merge(eco_quality, on='eco_id', how='left')

col_order = [
    'eco_id', 'eco_name', 'biome_num', 'biome_name', 'year',
    # Primary metrics
    'onset_doy', 'peak_doy', 'end_doy', 'season_length',
    'n_detections', 'onset_month', 'peak_month',
    # Alternative thresholds
    'onset_doy_10', 'end_doy_90',
    'onset_doy_15', 'end_doy_85',
    # Profile shape
    'median_doy', 'mean_median_div', 'q25_doy', 'q75_doy',
    'iqr_season_length', 'active_days',
    'peak_concentration', 'skewness', 'kurtosis_raw',
    # Bimodality diagnostics (per year)
    'bc', 'bimodal_flag_bc',
    'dip_stat', 'dip_pval', 'bimodal_flag_dip',
    'bimodal_flag_year',
    # Per-year quality
    'peak_outside_window', 'n_years_valid', 'pct_years_valid',
    # Ecoregion-level quality
    'cv_peak_doy', 'mean_profile_corr',
    'frac_flagged', 'bimodal_flag_eco',
    # Circular metrics (optional — handles year-boundary wrapping)
    'circ_mean_doy', 'circ_R', 'circ_rotated',
    'onset_doy_circ', 'end_doy_circ', 'season_len_circ',
]

col_order = [c for c in col_order if c in master_df.columns]
master_df = master_df[col_order]

master_path = os.path.join(output_dir, f'master_{_run_name}.csv')
master_df.to_csv(master_path, index=False)

print(f'Master CSV: {master_df.shape[0]} rows × {master_df.shape[1]} columns')
print(f'Path: {os.path.abspath(master_path)}')
print()
print(master_df.head(10).to_string())

Master CSV: 14885 rows × 44 columns
Path: C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v6\fire_metrics\master_global_v6.csv

   eco_id                        eco_name  biome_num                                      biome_name  year  onset_doy  peak_doy  end_doy  season_length  n_detections  onset_month  peak_month  onset_doy_10  end_doy_90  onset_doy_15  end_doy_85  median_doy  mean_median_div  q25_doy  q75_doy  iqr_season_length  active_days  peak_concentration  skewness  kurtosis_raw      bc  bimodal_flag_bc  dip_stat  dip_pval  bimodal_flag_dip  bimodal_flag_year  peak_outside_window  n_years_valid  pct_years_valid  cv_peak_doy  mean_profile_corr  frac_flagged  bimodal_flag_eco  circ_mean_doy  circ_R  circ_rotated  onset_doy_circ  end_doy_circ  season_len_circ
0       1  Albertine Rift montane forests          1  Tropical & Subtropical Moist Broadleaf Forests  2003         35       193      278            244         26772            2           7            95         257